# Session 1 Lab: Time-series forecasting

This lab shows how to turn a small news corpus into a simple macroeconomic signal.

We will:
1. identify which headlines are economically relevant
2. infer business sentiment with zero-shot methods
3. aggregate those article-level scores into a Business Confidence Index (BCI)
4. test whether BCI helps forecast private investment


In [ ]:
from FewShotX import Embeddings, ZeroShotLearner, ZeroShotNLI, evaluate_predictions
from FewShotX.notebook import configure_notebook
from utils.utils_meridia import (
    add_binary_predictions,
    build_bci_series,
    build_manual_ar_design,
    calculate_forecast_metrics,
    dynamic_correlation_plot,
    plot_bci_and_investment,
    plot_forecasts,
    print_forecast_metrics,
    rolling_autoregressive_forecast,
)

_, _ = configure_notebook(theme="whitegrid", progress_bar="rich")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Read the CSV 
df_corpus = pd.read_parquet("datasets/meridia_corpus.parquet")
df_corpus.head()


## 1. Economic Relevance

We begin by asking a simple question: *is this headline about the economy/finance?*

The first approach uses embedding similarity. The second uses NLI-based zero-shot classification.


In [ ]:
embedder = Embeddings(model_name="all-MiniLM-L6-v2")
df_embed = embedder.embed_df(df_corpus, text_col="headline")
df_embed.head()


In [ ]:
labels = ["This example is about economics or finance"]

zs1 = ZeroShotLearner(embedder.model, similarity="cosine")
scored_df1 = zs1.score_df(
    df=df_embed,
    text_embedding_cols=[f"emb_{i}" for i in range(embedder.embedding_dim)],
    labels=labels,
    label_names=["is_financial_zs1"],
)
scored_df1 = scored_df1[["date", "headline", "category", "is_financial", "is_financial_zs1"]]
scored_df1.head()


In [ ]:
zs2 = ZeroShotNLI()
scored_df2 = zs2.score_df(
    scored_df1,
    text_col="headline",
    labels=["about economics or finance"],
    label_names=["is_financial_zs2"],
)
scored_df2.head()


In [ ]:
scored_df2, thresholds = add_binary_predictions(
    scored_df2,
    score_columns={
        "is_financial_zs1": "pred_zs1",
        "is_financial_zs2": "pred_zs2",
    },
)

print(f"Optimal threshold for ZS1: {thresholds['pred_zs1']:.4f}")
print(f"Optimal threshold for ZS2: {thresholds['pred_zs2']:.4f}")
scored_df2.head()


In [ ]:
metrics = evaluate_predictions(
    y_true=scored_df2["is_financial"],
    ZS1=scored_df2["is_financial_zs1"],
    ZS2=scored_df2["is_financial_zs2"],
    return_metrics=True,
)
metrics


### Reading The Precision-Recall View

For this task, recall tells us how many truly economic headlines we recover, while precision tells us how many predicted economic headlines are actually correct.

That trade-off matters because a permissive threshold keeps more relevant news, but it also lets more non-economic headlines into the pipeline.


In [ ]:
scored_df2.loc[
    (scored_df2["is_financial"] == 0) & (scored_df2["pred_zs1"] == 1),
    ["headline", "is_financial_zs1", "pred_zs1"],
].head(10)


## 2. Build A Business Confidence Index

Once we know which headlines are economically relevant, we can score their tone as **positive**, **neutral**, or **negative** for business confidence.


In [ ]:
zs_sentiment = ZeroShotNLI()
scored_df3 = zs_sentiment.score_df(
    scored_df1,
    text_col="headline",
    labels=[
        "Positive business confidence",
        "Neutral business confidence",
        "Negative business confidence",
    ],
    label_names=["positive", "neutral", "negative"],
)
scored_df3.head()


In [ ]:
scored_df3["pred_zs1"] = (scored_df3["is_financial_zs1"] >= thresholds["pred_zs1"]).astype(int)
scored_df3, monthly_bci = build_bci_series(
    scored_df3,
    relevance_pred_col="pred_zs1",
    positive_col="positive",
    neutral_col="neutral",
    negative_col="negative",
)

cm = confusion_matrix(
    scored_df3["category"].str.lower(),
    scored_df3["max_sentiment"],
    labels=["positive", "neutral", "negative"],
)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Positive", "Neutral", "Negative"],
).plot(cmap="Blues", colorbar=True)
plt.title("Headline sentiment: ground truth vs predicted sentiment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(False)
plt.show()

monthly_bci.head()


## 3. Compare BCI With Private Investment

The next question is whether the text-based BCI co-moves with the target series and whether it appears to lead the cycle.


In [ ]:
df_investment = pd.read_parquet("datasets/meridia_investment.parquet")
df_investment["investment_growth"] = df_investment["investment_growth"] * 100
df_merge = pd.merge(monthly_bci, df_investment, on="date", how="left")
plot_bci_and_investment(df_merge)
df_merge.head()


## 4. From Signal To Forecast

We first inspect lead-lag correlations, then estimate a manual AR(1) benchmark, and finally compare recursive forecasts with and without the BCI.


In [ ]:
df_base = df_merge.copy()
df_base["date"] = df_base["date"].dt.to_timestamp()

best_lag, best_corr = dynamic_correlation_plot(df_base.copy(), "investment_growth", "BCI", max_lag=8)
Y, X = build_manual_ar_design(df_base, target_col="investment_growth", lags=1)
beta = np.linalg.solve(X.T @ X, X.T @ Y)

print(f"Best correlation lag: {best_lag}")
print(f"Best correlation: {best_corr:.2f}")
print(f"Number of observations: {len(Y)}")
pd.DataFrame(beta, index=["const", "investment_growth_lag1"], columns=["coefficient"])


In [ ]:
df_without_bci = rolling_autoregressive_forecast(
    df_base,
    start_date="2023-01",
    end_date="2025-04",
    lags=1,
)
df_with_bci = rolling_autoregressive_forecast(
    df_base,
    start_date="2023-01",
    end_date="2025-04",
    lags=1,
    exog_col="BCI",
)

plot_forecasts(df_base, df_without_bci, df_with_bci)

metrics_without_bci = calculate_forecast_metrics(df_base, df_without_bci)
metrics_with_bci = calculate_forecast_metrics(df_base, df_with_bci)

print_forecast_metrics(metrics_without_bci, "AR(1) without BCI")
print_forecast_metrics(metrics_with_bci, "AR(1) with BCI")
